In [1]:
import pandas as pd
import re

In [2]:
# pd.set_option('display.max_rows', 100)

In [3]:
pinyindic_df = pd.read_csv("character_dictionary_withEng.csv", usecols=["character","pinyin_givenname","pinyin_familyname","pinyin","english_givenname","english_familyname","english"])
pinyindic_df

,character,pinyin_givenname,pinyin_familyname,pinyin,english_givenname,english_familyname,english
0,一,yī,yī,yī,yi,yi,yi
1,乙,yǐ,yǐ,yǐ,yi,yi,yi
2,二,èr,èr,èr,er,er,er
3,十,shí,shí,shí,shi,shi,shi
4,丁,dīng#####zhēng,dīng,dīng#####zhēng,ding#####zheng,ding,ding#####zheng
...,...,...,...,...,...,...,...
8097,玃,jué,jué,jué,jue,jue,jue
8098,醾,mí,mí,mí,mi,mi,mi
8099,齇,zhā,zhā,zhā,zha,zha,zha
8100,觿,xī,xī,xī,xi,xi,xi


In [4]:
# special family name
family_py_df = pd.read_csv("family_py.csv")
family_py_df

,name,pinyin,english
0,曾,zēng,zeng
1,仇,qiú,qiu
2,朴,piáo,piao
3,单,shàn,shan
4,解,xiè,xie
...,...,...,...
66,慕容,mùróng,murong
67,鲜于,xiān yú,xianyu
68,闻人,wén rén,wenren
69,颛孙,zhuānsūn,zhuansun


In [5]:
# create dictionary for characters
pinyindic_df['english'] = pinyindic_df['english'].fillna('nan').astype(str)
pinyindic_df['english_givenname'] = pinyindic_df['english_givenname'].fillna('nan').astype(str)
pinyindic_df['english_familyname'] = pinyindic_df['english_familyname'].fillna('nan').astype(str)
char_to_pinyin_givenname = dict(zip(pinyindic_df['character'], pinyindic_df['pinyin_givenname']))
char_to_english_givenname = dict(zip(pinyindic_df['character'], pinyindic_df['english_givenname']))
char_to_pinyin_familyname = dict(zip(pinyindic_df['character'], pinyindic_df['pinyin_familyname']))
char_to_english_familyname = dict(zip(pinyindic_df['character'], pinyindic_df['english_familyname']))

# Iterate through the special family name and update the dictionary
for index, row in family_py_df.iterrows():
    name, pinyin, english = row['name'], row['pinyin'], row['english']
    if name not in char_to_pinyin_familyname:
        char_to_pinyin_familyname[name] = pinyin
    if name not in char_to_english_familyname:
        char_to_english_familyname[name] = english

# Function to get combined Pinyin for a Chinese name
def charas_to_pinyin_givenname(chinese_name,separator):
    chinese_name = str(chinese_name)
    if pd.notna(chinese_name):
        return separator.join([char_to_pinyin_givenname[char] for char in chinese_name if char in char_to_pinyin_givenname])
    else:
        return chinese_name
def charas_to_english_givenname(chinese_name,separator):
    chinese_name = str(chinese_name)
    if pd.notna(chinese_name):
        return separator.join([char_to_english_givenname[str(char)] for char in chinese_name if str(char) in char_to_english_givenname])
    else:
        return chinese_name

def charas_to_pinyin_familyname(chinese_name):
    chinese_name = str(chinese_name)
    if pd.notna(chinese_name):
        return char_to_pinyin_familyname[chinese_name]
    else:
        return chinese_name
def charas_to_english_familyname(chinese_name):
    chinese_name = str(chinese_name)
    if pd.notna(chinese_name):
        return char_to_english_familyname[chinese_name]
    else:
        return chinese_name
    

In [6]:
from itertools import product

def unfold_matrix(input_string):
    # Split the input string using ";;" as the delimiter
    sections = input_string.split(";;")

    # Generate combinations of elements from each split section
    combinations = product(*[section.split("#####") for section in sections])

    # Concatenate the combinations with ';'
    result = '#####'.join(''.join(combination) for combination in combinations)

    return result


In [13]:
CNKI_uniqueChineseNames = pd.read_csv('CNKI_uniqueChineseNames.csv')

In [14]:
# CNKI_uniqueChineseNames['familyname_pinyin'] = CNKI_uniqueChineseNames['familyname'].apply(charas_to_pinyin_familyname)
CNKI_uniqueChineseNames['familyname_english'] = CNKI_uniqueChineseNames['familyname'].apply(charas_to_english_familyname)

In [15]:
# CNKI_uniqueChineseNames['givenname_pinyin'] = CNKI_uniqueChineseNames['givenname'].apply(charas_to_pinyin_givenname,separator=";;")
CNKI_uniqueChineseNames['givenname_english'] = CNKI_uniqueChineseNames['givenname'].apply(charas_to_english_givenname,separator=";;")
# CNKI_uniqueChineseNames['givenname_pinyin'] = CNKI_uniqueChineseNames['givenname_pinyin'].apply(unfold_matrix)
CNKI_uniqueChineseNames['givenname_english'] = CNKI_uniqueChineseNames['givenname_english'].apply(unfold_matrix)

In [16]:
CNKI_uniqueChineseNames["familyname_english_first"] = CNKI_uniqueChineseNames["familyname_english"].str.split("#####").str[0]
CNKI_uniqueChineseNames["givenname_english_first"] = CNKI_uniqueChineseNames["givenname_english"].str.split("#####").str[0]

In [26]:
CNKI_uniqueChineseNames

,fullname,familyname,givenname,familyname_english,givenname_english,familyname_english_first,givenname_english_first,fullname_english
0,莫卫民,莫,卫民,mo,weimin,mo,weimin,moweimin
1,谭丽英,谭,丽英,tan,liying,tan,liying,tanliying
2,甄慧慧,甄,慧慧,zhen,huihui,zhen,huihui,zhenhuihui
3,陈醒基,陈,醒基,chen,xingji,chen,xingji,chenxingji
4,赵阶林,赵,阶林,zhao,jielin,zhao,jielin,zhaojielin
...,...,...,...,...,...,...,...,...
81192,邱岳明,邱,岳明,qiu,yueming,qiu,yueming,qiuyueming
81193,王亚平,王,亚平,wang,yaping,wang,yaping,wangyaping
81194,罗纯,罗,纯,luo,chun,luo,chun,luochun
81195,马雍翔,马,雍翔,ma,yongxiang,ma,yongxiang,mayongxiang


In [25]:
CNKI_uniqueChineseNames['fullname_english'] = CNKI_uniqueChineseNames['familyname_english_first'] + CNKI_uniqueChineseNames['givenname_english_first']

In [29]:
CNKI_uniqueChineseNames.to_csv('CNKI_uniqueChineseNames_withPinyin.csv', index = False)